# 02 — Data Preprocessing

## Project overview
This notebook prepares the dataset for NLP and machine learning. Candidate information becomes `candidate_text`, job information becomes `job_text`, and the processed dataset is saved for Notebooks 03–06.

**Output:** `processed_data.csv`.

In [37]:
import pandas as pd

from project_utils import clean_text

In [38]:
df=pd.read_csv(r"D:\Instant AI\AI\Graduation Project\My Project\Data\resume_data_for_ranking.csv")

In [39]:
df_processed = df.copy()

In [40]:
df_processed=df_processed.drop(columns=["responsibilities.1"])

## 1. Prepare the working dataset
Load the original data, create a working copy, remove duplicated fields, and exclude fields not needed for matching text.

In [41]:
print(df_processed.columns.tolist())

['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'age_requirement', 'skills_required', 'matched_score']


In [42]:
columns_to_exclude=["address",
                    "age_requirement",
                    "company_urls",
                    "extra_curricular_organization_links",
                    "online_links"
                    ]
df_processed=df_processed.drop(columns=columns_to_exclude,errors='ignore')

## 2. Handle missing text values
Text columns are filled with empty strings so later concatenation and NLP operations are safe.

In [43]:
df_processed.isnull().sum().sort_values(ascending=False)

proficiency_levels                     8844
languages                              8844
certification_providers                7536
certification_skills                   7536
issue_dates                            7536
expiry_dates                           7536
extra_curricular_activity_types        6118
extra_curricular_organization_names    6118
role_positions                         6118
career_objective                       4804
skills_required                        1701
experiencere_requirement               1364
educational_institution_name             84
educational_results                      84
positions                                84
end_dates                                84
related_skils_in_job                     84
professional_company_names               84
major_field_of_studies                   84
result_types                             84
start_dates                              84
degree_names                             84
passing_years                   

In [44]:
text_columns = df_processed.select_dtypes(include='object').columns

print(text_columns.tolist())

['career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'issue_dates', 'expiry_dates', 'job_position_name', 'educationaL_requirements', 'experiencere_requirement', 'skills_required']


C:\Users\mazen\AppData\Local\Temp\ipykernel_14308\2188934537.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df_processed.select_dtypes(include='object').columns


In [45]:
df_processed[text_columns] = df_processed[text_columns].fillna('')

In [46]:
df_processed[text_columns].isnull().sum().sum()

np.int64(0)

## 3. Clean and normalize text
The cleaning function lowercases text, removes repeated whitespace, and trims unnecessary spaces.

In [47]:
test_text="        PYthon            DEVeloper "
print(clean_text(test_text))

python developer


In [48]:
for column in text_columns:
    df_processed[column]=df_processed[column].apply(clean_text)

In [49]:
df_processed[['career_objective', 'skills', 'job_position_name']].head(5)

,career_objective,skills,job_position_name
0,big data analytics working and database wareho...,"['big data', 'hadoop', 'hive', 'python', 'mapr...",senior software engineer
1,fresher looking to join as a data analyst and ...,"['data analysis', 'data analytics', 'business ...",machine learning (ml) engineer
2,,"['software development', 'machine learning', '...","executive/ senior executive- trade marketing, ..."
3,to obtain a position in a fast-paced business ...,"['accounts payables', 'accounts receivables', ...",business development executive
4,professional accountant with an outstanding wo...,"['analytical reasoning', 'compliance testing k...",senior ios engineer


In [50]:
for column in text_columns:
    count_na=(df_processed[column]=="n/a").sum()
    count_none=(df_processed[column]=="none").sum()

    if count_na>0 or count_none>0:
        print(f"Column: {column}, Count of 'n/a': {count_na}, Count of 'none': {count_none}")

In [51]:
for column in text_columns:
    df_processed[column]=df_processed[column].replace(
        
        {
            "n/a": "",
            "none": ""
        }
    )

In [52]:
candidate_columns = [
    'career_objective',
    'skills',
    'educational_institution_name',
    'degree_names',
    'major_field_of_studies',
    'professional_company_names',
    'positions',
    'related_skils_in_job',
    'extra_curricular_activity_types',
    'extra_curricular_organization_names',
    'role_positions',
    'languages',
    'proficiency_levels',
    'certification_providers',
    'certification_skills'
]

## 4. Build candidate and job text
Candidate-related fields are combined into `candidate_text`; job-related fields are combined into `job_text`.

In [53]:
candidate_columns = [
    col for col in candidate_columns 
    if col in df_processed.columns]

In [54]:
df_processed["candidate_text"]=(
    df_processed[candidate_columns]
    .fillna('')
    .astype(str)
    .agg(' '.join, axis=1)
)

In [55]:
import re
df_processed["skills"] = df_processed["skills"].apply(
    lambda t: re.sub(r"'?skill \d+'?,?\s*", "", t)
)

In [56]:
print(df_processed["candidate_text"])

0       big data analytics working and database wareho...
1       fresher looking to join as a data analyst and ...
2        ['software development', 'machine learning', ...
3       to obtain a position in a fast-paced business ...
4       professional accountant with an outstanding wo...
                              ...                        
9539     ['mathematical modelling', 'machine learning'...
9540    expertise eda modeler. i like to learn what my...
9541    looking for roles related to application devel...
9542     ['machine learning', 'natural language proces...
9543     ['assisted living', 'interpersonal and commun...
Name: candidate_text, Length: 9544, dtype: str


In [57]:
job_columns = [
    'job_position_name',
    'educationaL_requirements',
    'experiencere_requirement',
    'responsibilities',
    'skills_required'
]

job_columns = [
    col for col in job_columns
    if col in df_processed.columns
]

In [58]:
df_processed['job_text'] = (
    df_processed[job_columns]
    .fillna('')
    .astype(str)
    .agg(' '.join, axis=1)
)

In [59]:
print(df_processed['job_text'])

0       senior software engineer b.sc in computer scie...
1       machine learning (ml) engineer m.sc in compute...
2       executive/ senior executive- trade marketing, ...
3       business development executive bachelor/honors...
4       senior ios engineer bachelor of science (bsc) ...
                              ...                        
9539    data engineer bachelor of science (bsc) 5 to 8...
9540    executive/ sr. executive -it bachelor of scien...
9541    executive - vat bba in accounting and finance ...
9542    asst. manager/ manger (administrative) bachelo...
9543    civil engineer b.sc in civil engineering from ...
Name: job_text, Length: 9544, dtype: str


In [60]:
df_processed["candidate_text"]=df_processed["candidate_text"].apply(clean_text)

df_processed["job_text"]=df_processed["job_text"].apply(clean_text)

In [61]:
df_processed[[
"candidate_text", "job_text","matched_score"
]].head(5)

,candidate_text,job_text,matched_score
0,big data analytics working and database wareho...,senior software engineer b.sc in computer scie...,0.850000
1,fresher looking to join as a data analyst and ...,machine learning (ml) engineer m.sc in compute...,0.750000
2,"['software development', 'machine learning', '...","executive/ senior executive- trade marketing, ...",0.416667
3,to obtain a position in a fast-paced business ...,business development executive bachelor/honors...,0.760000
4,professional accountant with an outstanding wo...,senior ios engineer bachelor of science (bsc) ...,0.650000


## 5. Validate the processed text
Examples, empty records, and text lengths are checked to confirm that preprocessing produced usable text.

In [62]:
empty_candidates=(
    df_processed["candidate_text"].str.strip()==""
).sum()

print("Empty candidate profiles:", empty_candidates)

Empty candidate profiles: 0


In [63]:
empty_jobs=(
    df_processed["job_text"].str.strip()==""
).sum()

print("Empty job postings:", empty_jobs)

Empty job postings: 0


In [64]:
df_processed["candidate_text_length"]=(
    df_processed["candidate_text"].str.len()
)

df_processed["job_text_length"]=(
    df_processed["job_text"].str.len()
)

## 6. Text-length analysis
Descriptive statistics and short examples help identify unusual records.

In [65]:
df_processed[
    ["candidate_text_length", "job_text_length"]
    ].describe()

,candidate_text_length,job_text_length
count,9544.000000,9544.000000
mean,949.539396,372.769279
std,577.924801,116.453340
min,152.000000,209.000000
25%,528.000000,307.000000
50%,753.000000,360.000000
75%,1280.000000,392.000000
max,3813.000000,750.000000


In [66]:
df_processed[
    ['candidate_text', 'candidate_text_length']
].sort_values('candidate_text_length').head(10)

,candidate_text,candidate_text_length
3857,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
5832,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
4766,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
6511,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
4760,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
1510,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
4642,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
9017,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
5453,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152
1174,"['skill 1', 'skill 3', 'skill 5', 'skill 2', '...",152


In [67]:
df_processed[
    ['job_text', 'job_text_length']
].sort_values('job_text_length').head(10)

,job_text,job_text_length
6968,business development executive bachelor/honors...,209
2957,business development executive bachelor/honors...,209
6952,business development executive bachelor/honors...,209
6986,business development executive bachelor/honors...,209
4785,business development executive bachelor/honors...,209
7788,business development executive bachelor/honors...,209
4461,business development executive bachelor/honors...,209
2542,business development executive bachelor/honors...,209
4413,business development executive bachelor/honors...,209
2573,business development executive bachelor/honors...,209


In [68]:
df_processed[
    ['candidate_text', 'job_text', 'matched_score']
].head()

,candidate_text,job_text,matched_score
0,big data analytics working and database wareho...,senior software engineer b.sc in computer scie...,0.850000
1,fresher looking to join as a data analyst and ...,machine learning (ml) engineer m.sc in compute...,0.750000
2,"['software development', 'machine learning', '...","executive/ senior executive- trade marketing, ...",0.416667
3,to obtain a position in a fast-paced business ...,business development executive bachelor/honors...,0.760000
4,professional accountant with an outstanding wo...,senior ios engineer bachelor of science (bsc) ...,0.650000


In [69]:
df_processed.to_csv(
    r'D:\Instant AI\AI\Graduation Project\My Project\Data\processed_data.csv',
    index=False
)

## 7. Save the processed dataset
The processed dataset is saved as `processed_data.csv`, the shared input for the following notebooks.

In [70]:
import os

file_path = r'D:\Instant AI\AI\Graduation Project\My Project\Data\processed_data.csv'

print("File exists:", os.path.exists(file_path))

File exists: True
